# Week 04 — PyTorch for Accelerated Training
## Complete Code Companion (Fundamentals-First Edition)
**NPTEL · Applied Accelerated Artificial Intelligence**  
**Dr. Satyajit Das, IIT Guwahati**

---

This notebook is the complete runnable companion to the Week 04 revised lecture slides.  
Every slide concept has at least one code cell. Run cells **top to bottom** — later cells depend on earlier definitions.

| Segment | Topic | Cells |
|---------|-------|-------|
| **S1** | Tensors, Autograd & Computational Graph | 01–12 |
| **S2** | Building Models with nn.Module | 13–21 |
| **S3** | The Standard Training Loop | 22–32 |
| **S4** | DataLoader & Data Pipeline | 33–43 |
| **S5** | First Steps in Performance | 44–52 |
| **E2E** | End-to-End: Full Training Run | 53–55 |

### Setup (run once)
```bash
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install torchviz matplotlib tqdm
```

In [ ]:
# ── CELL 00 · Environment verification ───────────────────────────────────────
import sys, subprocess, platform
import torch, torchvision

print('=' * 58)
print('  WEEK 04 · ENVIRONMENT CHECK')
print('=' * 58)
print(f'Python        : {sys.version.split()[0]}')
print(f'PyTorch       : {torch.__version__}')
print(f'torchvision   : {torchvision.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU           : {p.name}')
    print(f'VRAM          : {p.total_memory/1e9:.1f} GB')
    print(f'SM count      : {p.multi_processor_count}')
    print(f'CUDA version  : {torch.version.cuda}')
    print(f'cuDNN version : {torch.backends.cudnn.version()}')
    print(f'BF16 support  : {torch.cuda.is_bf16_supported()}')
    print(f'TF32 matmul   : {torch.backends.cuda.matmul.allow_tf32}')
    print(f'Flash SDPA    : {torch.backends.cuda.flash_sdp_enabled()}')
else:
    print('  No GPU — timings shown are CPU baselines only.')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nActive device : {DEVICE}')
print('\nAll checks passed — ready to run.')

---
## Segment 1 · PyTorch Tensors, Autograd & the Computational Graph
### Slides 3–9

In [ ]:
# ── CELL 01 · Creating Tensors — every constructor from the slide ────────────
import torch

# From data
t1 = torch.tensor([1.0, 2.0, 3.0])
t2 = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)

# From shape
zeros  = torch.zeros(3, 4)           # 3x4 matrix of 0.0
ones   = torch.ones(2, 3, 4)         # 2x3x4 tensor of 1.0
randn  = torch.randn(4, 5)           # standard normal
arange = torch.arange(0, 10, 2)      # [0,2,4,6,8]
lspace = torch.linspace(0, 1, 6)     # [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
eye    = torch.eye(3)                # 3x3 identity matrix

print('tensor from list        :', t1)
print('zeros(3,4)              :', zeros.shape)
print('ones(2,3,4)             :', ones.shape)
print('randn(4,5)              :', randn.shape)
print('arange(0,10,2)          :', arange)
print('linspace(0,1,6)         :', lspace)
print('eye(3)                  :\n', eye)

In [ ]:
# ── CELL 02 · Data Types (dtype) ─────────────────────────────────────────────
import torch

dtypes = {
    'float32 (default)' : torch.float32,
    'float16 (FP16)'    : torch.float16,
    'bfloat16 (BF16)'   : torch.bfloat16,
    'float64 (double)'  : torch.float64,
    'int64 (long)'      : torch.int64,
    'int32'             : torch.int32,
    'bool'              : torch.bool,
}

print(f'  {"dtype":<25}  {"bytes/elem":>10}  example value')
print('  ' + '-'*50)
for name, dt in dtypes.items():
    x = torch.tensor(1, dtype=dt)
    bpe = x.element_size()
    print(f'  {name:<25}  {bpe:>10}  {x.item()}')

# Conversion
x = torch.randn(3)
print(f'\nOriginal     : {x.dtype}')
print(f'to float16   : {x.to(torch.float16).dtype}')
print(f'to bfloat16  : {x.bfloat16().dtype}')
print(f'to int64     : {x.long().dtype}')

# INSTRUCTOR NOTE: Why does dtype matter?
# float32 = 4 bytes/element. A 7B-parameter model in FP32 = 7e9 * 4 = 28 GB
# In BF16 = 7e9 * 2 = 14 GB.  This is why mixed precision halves VRAM usage.
params = 7e9
print(f'\n7B params in FP32  : {params*4/1e9:.0f} GB')
print(f'7B params in BF16  : {params*2/1e9:.0f} GB')

In [ ]:
# ── CELL 03 · Device Placement ───────────────────────────────────────────────
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create on CPU
x_cpu = torch.randn(3, 3)
print(f'x_cpu.device  : {x_cpu.device}')

# Move to GPU
x_gpu = x_cpu.to(DEVICE)
print(f'x_gpu.device  : {x_gpu.device}')

# Create directly on GPU
y_gpu = torch.randn(3, 3, device=DEVICE)
print(f'y_gpu.device  : {y_gpu.device}')

# Trying to mix devices raises an error
try:
    z = x_cpu + x_gpu  # CPU + GPU
except RuntimeError as e:
    print(f'\nExpected error when mixing devices:')
    print(f'  {type(e).__name__}: {str(e)[:80]}')

# Correct: move to same device first
z = x_cpu.to(DEVICE) + x_gpu
print(f'\nAfter moving to same device: z.device = {z.device}')

# GPU memory info
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'\nGPU memory: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

In [ ]:
# ── CELL 04 · Tensor Operations: arithmetic, matmul, reductions ──────────────
import torch

A = torch.tensor([[1., 2.], [3., 4.]])
B = torch.tensor([[5., 6.], [7., 8.]])

print('A :\n', A)
print('B :\n', B)
print('\nElementwise A + B :\n', A + B)
print('Elementwise A * B :\n', A * B)
print('Matrix multiply A @ B :\n', A @ B)   # equivalent: torch.matmul(A, B)

# Reductions
x = torch.randn(3, 4)
print(f'\nx shape          : {x.shape}')
print(f'x.sum()          : {x.sum():.4f}      # scalar: sum all')
print(f'x.sum(dim=0)     : {x.sum(dim=0)}   # shape (4,): sum over rows')
print(f'x.sum(dim=1)     : {x.sum(dim=1)}  # shape (3,): sum over cols')
print(f'x.mean()         : {x.mean():.4f}')
print(f'x.max()          : {x.max():.4f}')
vals, idxs = x.max(dim=0)
print(f'x.max(dim=0)     values={vals}  indices={idxs}')

# Transpose
print(f'\nA.T shape        : {A.T.shape}   (row-major transpose)')
print(f'A.transpose(0,1) : {A.transpose(0,1).shape}')

In [ ]:
# ── CELL 05 · Reshaping, Indexing, Slicing ───────────────────────────────────
import torch

x = torch.arange(24, dtype=torch.float32)
print('arange(24) shape:', x.shape)   # (24,)

# Reshape
a = x.view(4, 6)          # shares memory — no copy
b = x.reshape(2, 3, 4)    # copies if not contiguous
print('view(4,6)         :', a.shape)
print('reshape(2,3,4)    :', b.shape)

# -1 infers size
c = x.view(6, -1)         # 6 rows, infer columns = 4
print('view(6,-1)        :', c.shape)

# unsqueeze / squeeze
d = a.unsqueeze(0)        # (4,6) -> (1,4,6)  adds batch dim
e = d.squeeze(0)          # (1,4,6) -> (4,6)
print('unsqueeze(0)      :', d.shape)
print('squeeze(0)        :', e.shape)

# permute — critical for images
img_hwc = torch.randn(224, 224, 3)   # PIL default: H,W,C
img_chw = img_hwc.permute(2, 0, 1)  # PyTorch default: C,H,W
print(f'\nPIL HWC -> PyTorch CHW: {img_hwc.shape} -> {img_chw.shape}')

# Indexing
m = torch.arange(12).view(3, 4).float()
print('\nm :\n', m)
print('m[0]         :', m[0])          # first row
print('m[-1]        :', m[-1])         # last row
print('m[1:3]       :\n', m[1:3])     # rows 1 and 2
print('m[:, 0]      :', m[:, 0])       # first column
print('m[m > 6]     :', m[m > 6])     # boolean mask

In [ ]:
# ── CELL 06 · Broadcasting — the rules with worked examples ──────────────────
import torch

# Rule: align from the right, dimension of 1 expands to match
print('Broadcasting examples:')
print('  (3,1) + (1,4) ->')
a = torch.ones(3, 1)
b = torch.ones(1, 4)
print(' ', (a + b).shape)             # (3,4)

print('  (4,) broadcast over (3,4) ->')
row_vec = torch.tensor([1., 2., 3., 4.])
matrix  = torch.ones(3, 4)
print(' ', (matrix + row_vec).shape)  # (3,4)

# Practical example: per-channel normalisation of an image batch
batch = torch.randn(8, 3, 224, 224)                     # NCHW
mean  = torch.tensor([0.485, 0.456, 0.406])             # shape (3,)
std   = torch.tensor([0.229, 0.224, 0.225])             # shape (3,)

# Must reshape to (1,3,1,1) to broadcast over N,H,W
mean = mean.view(1, 3, 1, 1)
std  = std.view(1, 3, 1, 1)
normalised = (batch - mean) / std
print(f'\nNormalised batch: {normalised.shape}')
print(f'Channel 0 mean after normalisation: {normalised[:,0].mean():.4f} (should be ~0)')

# Common mistake — wrong shape
print('\nCommon mistake:')
try:
    wrong = torch.ones(3, 4) + torch.ones(3)   # (3,4) + (3,) — FAILS
except RuntimeError as e:
    print(f'  Error: {str(e)[:60]}')
    print('  Fix: reshape (3,) to (3,1) so it aligns from the right')
fixed = torch.ones(3, 4) + torch.ones(3, 1)   # (3,4) + (3,1) -- OK
print(f'  Fixed shape: {fixed.shape}')

In [ ]:
# ── CELL 07 · Autograd — building and traversing the computational graph ─────
import torch

print('=' * 55)
print('  AUTOGRAD — HOW PYTORCH COMPUTES GRADIENTS')
print('=' * 55)

# Scalar example: L = w*x + w^2, compute dL/dw
w = torch.tensor(2.0, requires_grad=True)   # leaf tensor — tracked
x = torch.tensor(3.0)                       # input — not tracked

y = w * x + w**2                            # forward pass: builds graph
L = y.sum()                                 # scalar loss

print(f'w = {w.item()}')
print(f'x = {x.item()}')
print(f'L = w*x + w^2 = {L.item()}')
print(f'Expected dL/dw = x + 2w = {x.item()} + {2*w.item()} = {x.item()+2*w.item()}')

L.backward()                                # reverse pass: compute gradients
print(f'\nComputed w.grad  = {w.grad.item()}   ✓')

# Inspect the graph
print(f'\nTensor properties:')
print(f'  w.requires_grad : {w.requires_grad}')
print(f'  y.requires_grad : {y.requires_grad}  (propagated from w)')
print(f'  x.requires_grad : {x.requires_grad}  (not tracked)')
print(f'  w.is_leaf       : {w.is_leaf}  (we created it directly)')
print(f'  y.is_leaf       : {y.is_leaf}  (result of operation)')
print(f'  y.grad_fn       : {y.grad_fn}  (the op that created y)')

In [ ]:
# ── CELL 08 · Autograd on tensors (not just scalars) ─────────────────────────
import torch

# A simple linear layer from scratch
W = torch.randn(3, 2, requires_grad=True)   # weight matrix
b = torch.zeros(3,    requires_grad=True)   # bias vector
x = torch.randn(2)                          # input (no grad needed)

# Forward
z = W @ x + b          # linear transformation: shape (3,)
y = torch.relu(z)      # activation
L = y.sum()            # scalar loss (sum of outputs)

L.backward()

print('Manual linear layer forward + backward:')
print(f'  x shape   : {x.shape}')
print(f'  W shape   : {W.shape}')
print(f'  z = W@x+b : {z.shape}')
print(f'  y=relu(z) : {y.shape}')
print(f'  L         : {L.item():.4f}')
print(f'  W.grad    : {W.grad}   (shape {W.grad.shape})')
print(f'  b.grad    : {b.grad}')

# Demonstrate gradient accumulation
print('\nGradient accumulation demonstration:')
print(f'  b.grad before zero  : {b.grad.tolist()}')
(W @ x + b).sum().backward()   # second backward WITHOUT zeroing
print(f'  b.grad after 2nd bwd: {b.grad.tolist()}  <- gradients ACCUMULATED')
b.grad.zero_()                 # manual zero
print(f'  b.grad after zero_(): {b.grad.tolist()}  <- cleared')

In [ ]:
# ── CELL 09 · torch.no_grad, detach, retain_graph ────────────────────────────
import torch

w = torch.tensor(2.0, requires_grad=True)

# 1. torch.no_grad() — block-level, no graph built at all
with torch.no_grad():
    y = w * 3.0
    print(f'Inside no_grad: y.requires_grad = {y.requires_grad}  (False — no graph)')

# 2. .detach() — tensor-level, break gradient flow
y = w * 3.0
y_detached = y.detach()
print(f'y.requires_grad          : {y.requires_grad}')
print(f'y_detached.requires_grad : {y_detached.requires_grad}  (detached from graph)')
print(f'y_detached shares data   : {y_detached.data_ptr() == y.data_ptr()}  (same memory)')

# 3. retain_graph=True — keep graph after backward
w2 = torch.tensor(1.0, requires_grad=True)
y2 = w2 ** 3
L1 = y2 * 2
L2 = y2 * 5
L1.backward(retain_graph=True)   # keep graph alive
print(f'\nAfter L1.backward (retain): w2.grad = {w2.grad.item():.1f}  (= 2 * 3w^2 = 6)')
L2.backward()                    # now use graph for L2
print(f'After L2.backward         : w2.grad = {w2.grad.item():.1f}  (accumulated: 6 + 15 = 21)')

print('\nUse cases:')
print('  no_grad  -> inference, validation, eval loops')
print('  detach   -> freeze an encoder, stop gradients in GAN discriminator update')
print('  retain_graph -> two losses sharing a forward pass (GAN, multi-task learning)')

In [ ]:
# ── CELL 10 · NumPy interop ───────────────────────────────────────────────────
import torch
import numpy as np

# NumPy -> PyTorch (zero-copy, shared memory)
np_arr = np.array([1.0, 2.0, 3.0])
t      = torch.from_numpy(np_arr)

print('numpy -> torch (shared memory):')
print(f'  np_arr: {np_arr},  t: {t}')
np_arr[0] = 99.0   # modify numpy
print(f'  After np_arr[0]=99:  t = {t}  (changed!)')

# PyTorch -> NumPy (zero-copy, CPU only)
t2     = torch.tensor([4.0, 5.0, 6.0])
np_out = t2.numpy()
print(f'\ntorch -> numpy: {np_out}')
t2[1] = 500.0
print(f'After t2[1]=500: np_out = {np_out}  (also changed!)')

# GPU tensor: must .cpu() first
if torch.cuda.is_available():
    t_gpu = torch.randn(3, device='cuda')
    try:
        t_gpu.numpy()
    except (RuntimeError, TypeError) as e:
        print(f'\nGPU tensor .numpy() error: {str(e)[:50]}')
    np_from_gpu = t_gpu.cpu().numpy()   # correct
    print(f'GPU -> cpu -> numpy: {np_from_gpu}')

In [ ]:
# ── CELL 11 · Dynamic computational graph demonstration ──────────────────────
import torch

# Dynamic graph: different shapes each call, Python if/else works naturally
def flexible_forward(x, use_squared=True):
    """
    The graph this function builds changes depending on use_squared.
    In static frameworks (TF1), this would require conditional ops.
    In PyTorch, it just works.
    """
    if use_squared:             # Python control flow inside forward
        return (x ** 2).sum()
    else:
        return x.abs().sum()

w = torch.randn(4, requires_grad=True)

# Path 1: squared path
L1 = flexible_forward(w, use_squared=True)
L1.backward()
grad_squared = w.grad.clone()
w.grad.zero_()

# Path 2: abs path — different graph!
L2 = flexible_forward(w, use_squared=False)
L2.backward()
grad_abs = w.grad.clone()

print('Dynamic graph — two different computational paths:')
print(f'  w           : {w.detach().tolist()}')
print(f'  grad (x^2)  : {grad_squared.tolist()}   (= 2w)')
print(f'  grad (|x|)  : {grad_abs.tolist()}   (= sign(w))')
print('\nBoth computed with .backward() — no code change needed.')
print('This is the dynamic graph advantage over static frameworks.')

In [ ]:
# ── CELL 12 · Segment 1 Review: build a tiny network from raw tensors ────────
# This shows EXACTLY what nn.Module will hide from you in Segment 2.
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# ── Manually define a 2-layer MLP ────────────────────────────────────────────
D_in, D_hidden, D_out = 4, 8, 2

# CORRECT initialisation pattern:
#   torch.randn(..., requires_grad=True) * 0.1  is WRONG —
#   the * 0.1 multiplication produces a non-leaf tensor (it has a grad_fn),
#   so .grad is never populated and training crashes.
#   Always initialise the data FIRST, then attach requires_grad.
W1 = (torch.randn(D_hidden, D_in)  * 0.1).detach().requires_grad_(True)
b1 =  torch.zeros(D_hidden                ).requires_grad_(True)
W2 = (torch.randn(D_out, D_hidden) * 0.1).detach().requires_grad_(True)
b2 =  torch.zeros(D_out                   ).requires_grad_(True)

# Sanity check — all four must be leaf tensors
for name, p in [('W1', W1), ('b1', b1), ('W2', W2), ('b2', b2)]:
    assert p.is_leaf, f"{name} is NOT a leaf tensor — fix initialisation!"
    assert p.requires_grad, f"{name} does not require grad!"
print("All parameters are leaf tensors with requires_grad=True ✓")

params = [W1, b1, W2, b2]
lr = 0.01

# Fake batch
x = torch.randn(16, D_in)
y = torch.randint(0, D_out, (16,))

print("\nManual 2-layer MLP training for 10 steps:")
for step in range(10):
    # ── Forward ───────────────────────────────────────────────────────────────
    h      = F.relu(x @ W1.T + b1)
    logits = h @ W2.T + b2
    loss   = F.cross_entropy(logits, y)

    # ── Backward ──────────────────────────────────────────────────────────────
    loss.backward()

    # ── Manual SGD update ─────────────────────────────────────────────────────
    # Use torch.no_grad() to prevent gradient tracking on the update itself.
    # Use p.grad.data to access the raw gradient tensor safely.
    with torch.no_grad():
        for p in params:
            p.data.add_(p.grad.data, alpha=-lr)   # p = p - lr * grad

    # ── Zero gradients AFTER the update ───────────────────────────────────────
    # (alternatively call this at the TOP of the loop before forward — both work)
    for p in params:
        p.grad = None    # sets to None rather than zeroing — slightly faster

    if step % 2 == 0:
        print(f"  Step {step:2d}  loss = {loss.item():.4f}")

print("\nThis works — but we had to manually:")
print("  - initialise weights as proper leaf tensors (.detach().requires_grad_())")
print("  - write the forward pass with raw matmul")
print("  - call loss.backward() ourselves")
print("  - update weights manually with .data.add_(grad, alpha=-lr)")
print("  - zero gradients manually every step")
print("nn.Module + torch.optim handle all of this automatically.")


---
## Segment 2 · Building Neural Networks with nn.Module
### Slides 10–16

In [ ]:
# ── CELL 13 · The nn.Module contract ─────────────────────────────────────────
import torch
import torch.nn as nn

class MyFirstLayer(nn.Module):
    """
    The two required components:
      __init__  : define sub-layers as attributes
      forward   : define the computation
    """
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()              # ALWAYS call this first
        self.linear = nn.Linear(in_dim, out_dim)
        self.relu   = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(self.linear(x))  # call layers as functions

layer = MyFirstLayer(4, 3)
x     = torch.randn(2, 4)
out   = layer(x)              # calls __call__ which calls forward

print('MyFirstLayer:')
print(f'  Input  shape : {x.shape}')
print(f'  Output shape : {out.shape}')
print(f'  Output       : {out}')
print(f'\nModule tree:\n{layer}')

# What nn.Module provides
print('\nWhat nn.Module provides automatically:')
all_params = list(layer.parameters())
print(f'  layer.parameters()  : {len(all_params)} tensors')
for name, p in layer.named_parameters():
    print(f'    {name:<25} shape={p.shape}')
print(f'  layer.training mode  : {layer.training}  (True = train mode)')
layer.eval()
print(f'  after .eval()        : {layer.training}  (False = eval mode)')
layer.train()
print(f'  after .train()       : {layer.training}  (back to True)')

In [ ]:
# ── CELL 14 · Built-in layers reference ──────────────────────────────────────
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
B = 4  # batch size for demos

# ── Linear ───────────────────────────────────────────────────────────────────
linear = nn.Linear(512, 256)                    # W: (256,512), b: (256,)
x = torch.randn(B, 512)
print(f'Linear(512,256):  {x.shape} -> {linear(x).shape}')

# ── Convolutional ────────────────────────────────────────────────────────────
conv = nn.Conv2d(3, 64, kernel_size=3, padding=1)  # same spatial size
img  = torch.randn(B, 3, 32, 32)                   # NCHW
print(f'Conv2d(3,64,3,p=1): {img.shape} -> {conv(img).shape}')

# ── Normalisation ─────────────────────────────────────────────────────────────
bn = nn.BatchNorm2d(64)                    # C=64 feature maps
ln = nn.LayerNorm(256)                     # normalise last 256 dims
feat_map = torch.randn(B, 64, 8, 8)
print(f'BatchNorm2d(64):  {feat_map.shape} -> {bn(feat_map).shape}')
seq = torch.randn(B, 32, 256)              # (batch, seq_len, d_model)
print(f'LayerNorm(256):   {seq.shape} -> {ln(seq).shape}')

# ── Activations ──────────────────────────────────────────────────────────────
x_act = torch.randn(4)
print(f'\nReLU    : {nn.ReLU()(x_act)}')
print(f'GELU    : {nn.GELU()(x_act)}')
print(f'SiLU    : {nn.SiLU()(x_act)}')

# ── Embedding ────────────────────────────────────────────────────────────────
vocab_size, embed_dim = 10000, 128
emb     = nn.Embedding(vocab_size, embed_dim)
tok_ids = torch.randint(0, vocab_size, (B, 32))   # (batch, seq_len)
print(f'\nEmbedding(10000,128): {tok_ids.shape} -> {emb(tok_ids).shape}')

# ── Dropout ──────────────────────────────────────────────────────────────────
drop = nn.Dropout(p=0.3)
x_d  = torch.ones(10)
drop.train()   # active in train mode
print(f'\nDropout(0.3) train: {drop(x_d)}')
drop.eval()    # identity in eval mode
print(f'Dropout(0.3) eval : {drop(x_d)}')

In [ ]:
# ── CELL 15 · nn.Sequential vs Custom with skip connections ──────────────────
import torch
import torch.nn as nn

# ── 1. nn.Sequential — simple stacking ───────────────────────────────────────
mlp = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.BatchNorm1d(256),
    nn.Dropout(0.3),
    nn.Linear(256, 10),
)
x = torch.randn(8, 784)
print(f'Sequential MLP: {x.shape} -> {mlp(x).shape}')
print(mlp)

# ── 2. Custom Module with residual connection ─────────────────────────────────
class ResBlock(nn.Module):
    """A Pre-LN residual block (standard for modern Transformers)."""
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.net  = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-LN: normalise before the sub-network, add residual after
        return x + self.net(self.norm(x))  # skip connection: + x

block = ResBlock(d_model=128)
x_seq = torch.randn(4, 16, 128)   # (batch, seq_len, d_model)
out   = block(x_seq)
print(f'\nResBlock: {x_seq.shape} -> {out.shape}  (shape preserved by skip)')

# ── 3. nn.ModuleList — dynamic collections ────────────────────────────────────
class TransformerStack(nn.Module):
    def __init__(self, n_layers=4, d_model=128):
        super().__init__()
        # CORRECT: nn.ModuleList — PyTorch tracks these
        self.layers = nn.ModuleList([ResBlock(d_model) for _ in range(n_layers)])
        # WRONG: self.layers = [ResBlock(d_model) for _ in range(n_layers)]
        # A plain Python list is not tracked — parameters would be invisible!

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

stack = TransformerStack(n_layers=4, d_model=128)
print(f'\nTransformerStack(4 layers): {stack(x_seq).shape}')
n = sum(p.numel() for p in stack.parameters())
print(f'Parameters visible to PyTorch: {n:,}  (would be 0 with plain list!)')

In [ ]:
# ── CELL 16 · Inspecting model parameters ─────────────────────────────────
# NOTE: This cell is self-contained — ResBlock is redefined here.
import torch
import torch.nn as nn

# Inline ResBlock (also defined in Cell 15)
class ResBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.net  = nn.Sequential(
            nn.Linear(d_model, d_model*4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model*4, d_model)
        )
    def forward(self, x): return x + self.net(self.norm(x))

class InspectableModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed  = nn.Embedding(1000, 64)
        self.layers = nn.ModuleList([ResBlock(64) for _ in range(3)])
        self.head   = nn.Linear(64, 10)

    def forward(self, x):
        e = self.embed(x)
        for l in self.layers: e = l(e)
        return self.head(e.mean(1))

model = InspectableModel()

# ── 1. Parameter count ─────────────────────────────────────────────────────────
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total:,}')
print(f'Trainable parameters: {trainable:,}')

# ── 2. Named parameters ─────────────────────────────────────────────────────────
print('\nNamed parameters (first 8):')
for i, (name, p) in enumerate(model.named_parameters()):
    if i >= 8: print('  ...'); break
    print(f'  {name:<40} shape={list(p.shape)}')

# ── 3. Module tree ──────────────────────────────────────────────────────────────
print('\nModule tree:')
print(model)

# ── 4. Freeze specific layers ───────────────────────────────────────────────────
print('\nFreezing embedding layer:')
for p in model.embed.parameters():
    p.requires_grad = False
trainable_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Trainable after freeze: {trainable_after:,}')
print(f'  Frozen params         : {total - trainable_after:,}')


In [ ]:
# ── CELL 17 · Save and load model weights ────────────────────────────────────
import torch
import torch.nn as nn
import os, tempfile

# Build a small model
model = nn.Sequential(
    nn.Linear(8, 16), nn.GELU(), nn.Linear(16, 4)
)
x = torch.randn(2, 8)
out_before = model(x).detach().clone()

# ── Save state_dict (CORRECT pattern) ─────────────────────────────────────────
tmpdir = tempfile.mkdtemp()
path   = os.path.join(tmpdir, 'model_weights.pth')
torch.save(model.state_dict(), path)
print(f'Saved state_dict to: {path}')
print(f'File size: {os.path.getsize(path)} bytes')

# Inspect what state_dict contains
sd = model.state_dict()
print('\nstate_dict keys:')
for k, v in sd.items():
    print(f'  {k:<30} shape={list(v.shape)}')

# ── Load into fresh model ─────────────────────────────────────────────────────
model2 = nn.Sequential(
    nn.Linear(8, 16), nn.GELU(), nn.Linear(16, 4)
)
# Before loading: weights are random, outputs differ
out_before_load = model2(x).detach()

# Load
model2.load_state_dict(torch.load(path, weights_only=True))
out_after_load = model2(x).detach()

print(f'\nOutputs match after load: {torch.allclose(out_before, out_after_load)}')
print(f'Max diff: {(out_before - out_after_load).abs().max().item():.2e}')

# ── Weight initialisation with model.apply ────────────────────────────────────
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

model2.apply(init_weights)
print('\nWeight initialisation applied with model.apply(init_weights)')

In [ ]:
# ── CELL 18 · Complete model: MLP Classifier ─────────────────────────────────
import torch
import torch.nn as nn

class MLPClassifier(nn.Module):
    """
    3-layer MLP for multi-class classification.
    Uses Pre-LN pattern: LayerNorm -> Linear -> GELU -> Dropout.
    """
    def __init__(
        self,
        input_dim:  int,
        hidden_dim: int,
        output_dim: int,
        dropout:    float = 0.1,
    ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,  hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

model = MLPClassifier(input_dim=784, hidden_dim=512, output_dim=10)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(DEVICE)

# Test forward pass
x_test = torch.randn(16, 784, device=DEVICE)
logits = model(x_test)

print('MLPClassifier(784 -> 512 -> 512 -> 10):')
print(f'  Input  : {x_test.shape}')
print(f'  Output : {logits.shape}  (raw logits, NOT probabilities)')
total = sum(p.numel() for p in model.parameters())
print(f'  Params : {total:,}  ({total/1e6:.2f}M)')
print()
print(model)

In [ ]:
# ── CELL 19 · Complete model: Mini Transformer ───────────────────────────────
import torch
import torch.nn as nn

class TextClassifier(nn.Module):
    """
    Mini Transformer for binary/multi-class text classification.
    Architecture: Embedding -> TransformerEncoderLayers -> mean pool -> Linear head.
    """
    def __init__(
        self,
        vocab_size:   int = 30000,
        d_model:      int = 128,
        n_heads:      int = 4,
        n_layers:     int = 2,
        d_ff:         int = 512,
        num_classes:  int = 2,
        dropout:      float = 0.1,
        max_seq_len:  int = 256,
    ):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.layers  = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model       = d_model,
                nhead         = n_heads,
                dim_feedforward = d_ff,
                dropout       = dropout,
                activation    = 'gelu',
                batch_first   = True,     # input is (B, T, d_model)
                norm_first    = True,     # Pre-LN: more stable
            )
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        B, T   = token_ids.shape
        pos    = torch.arange(T, device=token_ids.device).unsqueeze(0)  # (1,T)
        x      = self.embed(token_ids) + self.pos_emb(pos)  # (B,T,d)
        for layer in self.layers:
            x = layer(x)
        x      = self.norm(x)
        pooled = x.mean(dim=1)          # mean-pool over sequence: (B,d)
        return self.head(pooled)        # (B, num_classes)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = TextClassifier().to(DEVICE)

token_ids = torch.randint(1, 30000, (8, 64), device=DEVICE)  # (B, T)
logits    = model(token_ids)

print('TextClassifier:')
print(f'  Input  : {token_ids.shape}')
print(f'  Output : {logits.shape}')
total = sum(p.numel() for p in model.parameters())
print(f'  Params : {total:,}')
print()
print(model)

In [ ]:
# ── CELL 20 · Demonstrating train vs eval mode ────────────────────────────────
import torch
import torch.nn as nn

# Dropout changes behaviour between train and eval
model_demo = nn.Sequential(
    nn.Linear(8, 16),
    nn.Dropout(p=0.5),   # drops 50% of activations in train mode
    nn.Linear(16, 4),
)
x = torch.ones(1, 8)

# Train mode
model_demo.train()
out_train_1 = model_demo(x).detach()
out_train_2 = model_demo(x).detach()

# Eval mode
model_demo.eval()
out_eval_1  = model_demo(x).detach()
out_eval_2  = model_demo(x).detach()

print('Effect of model.train() vs model.eval() on Dropout(0.5):')
print(f'  train mode run 1 : {out_train_1.tolist()}')
print(f'  train mode run 2 : {out_train_2.tolist()}  <- DIFFERENT (random dropout)')
print(f'  eval  mode run 1 : {out_eval_1.tolist()}')
print(f'  eval  mode run 2 : {out_eval_2.tolist()}  <- IDENTICAL (dropout disabled)')
print()
print('IMPORTANT: Always call model.eval() before validation/inference!')
print('           Always call model.train() at the start of each training epoch!')

In [ ]:
# ── CELL 21 · Segment 2 review: build CNN for CIFAR-10 ───────────────────────
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    """
    A 4-layer CNN for 32x32 image classification (CIFAR-10 style).
    Demonstrates: Conv2d, BatchNorm2d, MaxPool2d, Linear, skip-free architecture.
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3->32 channels, 32x32->16x16
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),             # 32x32 -> 16x16

            # Block 2: 32->64 channels, 16x16->8x8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),             # 16x16 -> 8x8

            # Block 3: 64->128 channels, 8x8->4x4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),             # 8x8 -> 4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                # (B,128,4,4) -> (B,2048)
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cnn    = SimpleCNN().to(DEVICE)
imgs   = torch.randn(8, 3, 32, 32, device=DEVICE)
out    = cnn(imgs)

print(f'SimpleCNN: {imgs.shape} -> {out.shape}')
total = sum(p.numel() for p in cnn.parameters())
print(f'Parameters: {total:,}')
print(cnn)

---
## Segment 3 · The Standard Training Loop
### Slides 17–23

In [ ]:
# ── CELL 22 · The six-step training loop (bare minimum) ──────────────────────
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Setup
model     = MLPClassifier(784, 256, 10).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

# Fake data
X = torch.randn(512, 784)
Y = torch.randint(0, 10, (512,))
dataset    = torch.utils.data.TensorDataset(X, Y)
loader     = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

print('THE SIX-STEP TRAINING LOOP')
print('=' * 50)
print('Step 1: model.train()       — activate Dropout/BatchNorm')
print('Step 2: zero_grad()         — clear old gradients')
print('Step 3: forward pass        — build computational graph')
print('Step 4: compute loss        — scalar measure of error')
print('Step 5: loss.backward()     — compute all gradients')
print('Step 6: optimizer.step()    — update weights')
print('=' * 50)

model.train()                              # Step 1
for epoch in range(3):
    epoch_loss = 0.0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()              # Step 2
        preds = model(batch_x)             # Step 3
        loss  = criterion(preds, batch_y)  # Step 4
        loss.backward()                    # Step 5
        optimizer.step()                   # Step 6

        epoch_loss += loss.item()
    print(f'Epoch {epoch+1}  avg loss: {epoch_loss/len(loader):.4f}')

print('\nTraining complete.')

In [ ]:
# ── CELL 23 · Demonstrating the 3 most common training loop bugs ─────────────
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = MLPClassifier(64, 64, 4).to(DEVICE)
opt    = torch.optim.SGD(model.parameters(), lr=0.01)
crit   = nn.CrossEntropyLoss()
x      = torch.randn(8, 64, device=DEVICE)
y      = torch.randint(0, 4, (8,), device=DEVICE)

# ── Bug 1: Forgetting zero_grad ───────────────────────────────────────────────
print('Bug 1: Forgetting zero_grad()')
model.zero_grad()
for i in range(3):
    loss = crit(model(x), y)
    loss.backward()                    # grad accumulates each step!
    if i == 0: g0 = model.net[0].weight.grad.norm().item()
    if i == 2: g2 = model.net[0].weight.grad.norm().item()

print(f'  grad norm after step 1: {g0:.4f}')
print(f'  grad norm after step 3: {g2:.4f}  <- 3x larger (accumulated!)')
print(f'  Fix: call zero_grad() BEFORE backward in every step')

# ── Bug 2: Calling model.forward(x) directly ─────────────────────────────────
print('\nBug 2: model.forward(x) vs model(x)')
hooks_called = []
def hook(module, inp, out):
    hooks_called.append('hook_called')
h = model.register_forward_hook(hook)

hooks_called.clear()
_ = model.forward(x)          # bypasses __call__
print(f'  model.forward(x) -> hooks called: {len(hooks_called)}  (ZERO — hooks bypassed!)')

hooks_called.clear()
_ = model(x)                  # goes through __call__
print(f'  model(x)         -> hooks called: {len(hooks_called)}  (correct)')
h.remove()

# ── Bug 3: Data not on GPU ────────────────────────────────────────────────────
print('\nBug 3: Data on wrong device')
x_cpu = torch.randn(4, 64)   # on CPU
try:
    _ = model(x_cpu)          # model on GPU, data on CPU
except RuntimeError as e:
    print(f'  Error: {str(e)[:70]}')
    print(f'  Fix: x = x.to(device) at start of every training step')

In [ ]:
# ── CELL 24 · Optimisers: SGD, Adam, AdamW with visual comparison ────────────
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_n_steps(optimizer_class, optimizer_kwargs, n_steps=100):
    torch.manual_seed(42)
    model = MLPClassifier(64, 64, 4).to(DEVICE)
    opt   = optimizer_class(model.parameters(), **optimizer_kwargs)
    crit  = nn.CrossEntropyLoss()
    losses = []
    for _ in range(n_steps):
        x = torch.randn(32, 64, device=DEVICE)
        y = torch.randint(0, 4, (32,), device=DEVICE)
        model.train()
        opt.zero_grad()
        loss = crit(model(x), y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

configs = [
    (torch.optim.SGD,   {'lr': 0.01, 'momentum': 0.9, 'weight_decay': 1e-4}, 'SGD+momentum'),
    (torch.optim.Adam,  {'lr': 3e-4, 'weight_decay': 1e-4},                  'Adam'),
    (torch.optim.AdamW, {'lr': 3e-4, 'weight_decay': 0.1},                   'AdamW'),
]

print(f'  {"Optimiser":<15}  {"Loss@10":>8}  {"Loss@50":>8}  {"Loss@100":>8}')
print('  ' + '-'*44)
for opt_cls, kwargs, name in configs:
    L = train_n_steps(opt_cls, kwargs)
    print(f'  {name:<15}  {L[9]:>8.4f}  {L[49]:>8.4f}  {L[99]:>8.4f}')

print('\nKey differences:')
print('  SGD   : needs careful LR tuning; best final accuracy on vision tasks')
print('  Adam  : adaptive LR; less tuning; weight decay is not properly decoupled')
print('  AdamW : correct decoupled weight decay; use for Transformers and LLMs')

In [ ]:
# ── CELL 25 · Learning rate schedulers ───────────────────────────────────────
import torch

model  = MLPClassifier(64, 64, 4)
n_steps, n_epochs = 100, 10

# ── Cosine Annealing ─────────────────────────────────────────────────────────
opt1  = torch.optim.AdamW(model.parameters(), lr=3e-4)
sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=n_epochs)
lrs_cosine = []
for e in range(n_epochs):
    lrs_cosine.append(opt1.param_groups[0]['lr'])
    sched1.step()   # call once per epoch

# ── OneCycleLR (warm-up + cosine decay) ──────────────────────────────────────
opt2   = torch.optim.AdamW(model.parameters(), lr=3e-4)
sched2 = torch.optim.lr_scheduler.OneCycleLR(
    opt2, max_lr=3e-4, total_steps=n_steps
)
lrs_one = []
for s in range(n_steps):
    lrs_one.append(opt2.param_groups[0]['lr'])
    sched2.step()   # call once per STEP (not epoch) for OneCycleLR

print('Learning Rate Schedule Summary')
print(f'{"Epoch":<8}  CosineAnneal')
for i, lr in enumerate(lrs_cosine):
    print(f'  {i+1:<6}  {lr:.6f}')

print(f'\nOneCycleLR — first 10 and last 5 steps:')
for i in list(range(10)) + list(range(-5, 0)):
    print(f'  Step {i if i>=0 else n_steps+i+1:<4}  lr={lrs_one[i]:.6f}')

print('\nNote: CosineAnnealingLR is called per epoch.')
print('      OneCycleLR is called per STEP (after every optimizer.step()).')

In [ ]:
# ── CELL 26 · Loss functions: every loss from the slide with worked examples ──
import torch
import torch.nn as nn
import torch.nn.functional as F

print('=' * 55)
print('  LOSS FUNCTIONS — MATCHED TO TASKS')
print('=' * 55)

# ── 1. CrossEntropyLoss — multi-class classification ──────────────────────────
crit_ce = nn.CrossEntropyLoss(label_smoothing=0.1)
logits  = torch.tensor([[2.0, 0.5, -1.0],   # sample 1 prefers class 0
                         [-0.5, 3.0, 1.0]])  # sample 2 prefers class 1
labels  = torch.tensor([0, 1])               # integer class IDs
loss_ce = crit_ce(logits, labels)
print(f'CrossEntropyLoss (multi-class)  : {loss_ce.item():.4f}')
print(f'  logits shape: {logits.shape}, labels shape: {labels.shape}')
print(f'  Takes RAW LOGITS — applies log_softmax internally. Never pass softmax!')

# ── 2. BCEWithLogitsLoss — binary classification ──────────────────────────────
crit_bce = nn.BCEWithLogitsLoss()
logits_b = torch.tensor([2.5, -1.2, 0.8, -0.3])  # raw scores (no sigmoid)
labels_b = torch.tensor([1.0, 0.0, 1.0, 0.0])     # float targets: 1=positive, 0=negative
loss_bce = crit_bce(logits_b, labels_b)
print(f'\nBCEWithLogitsLoss (binary)      : {loss_bce.item():.4f}')
print(f'  Also takes RAW LOGITS — applies sigmoid internally.')

# ── 3. MSELoss — regression ───────────────────────────────────────────────────
crit_mse  = nn.MSELoss()
crit_l1   = nn.L1Loss()
crit_hub  = nn.HuberLoss(delta=1.0)
preds     = torch.tensor([1.2, 3.5, 2.8, 4.1])
targets   = torch.tensor([1.0, 4.0, 2.5, 5.0])
print(f'\nMSELoss   : {crit_mse(preds, targets).item():.4f}  (squares errors — sensitive to outliers)')
print(f'L1Loss    : {crit_l1(preds, targets).item():.4f}  (abs errors  — robust to outliers)')
print(f'HuberLoss : {crit_hub(preds, targets).item():.4f}  (L2 near 0, L1 far   — best of both)')

# ── 4. Language modelling — reshaped CE with ignore_index ─────────────────────
B, T, V = 2, 8, 100    # batch, seq_len, vocab_size
crit_lm  = nn.CrossEntropyLoss(ignore_index=0)   # 0 = padding token
lm_logits = torch.randn(B, T, V)
lm_labels = torch.randint(0, V, (B, T))
lm_labels[0, 5:] = 0    # simulate padding in second half
# Reshape: (B,T,V) -> (B*T,V)   and   (B,T) -> (B*T,)
loss_lm = crit_lm(lm_logits.view(-1, V), lm_labels.view(-1))
print(f'\nLM CrossEntropyLoss (ignore pad): {loss_lm.item():.4f}')
print(f'  Reshape: ({B},{T},{V}) -> ({B*T},{V})  and  ({B},{T}) -> ({B*T},)')

In [ ]:
# ── CELL 27 · Gradient clipping ──────────────────────────────────────────────
import torch
import torch.nn as nn

model = MLPClassifier(64, 64, 4)
opt   = torch.optim.AdamW(model.parameters(), lr=3e-4)
crit  = nn.CrossEntropyLoss()
x     = torch.randn(8, 64); y = torch.randint(0, 4, (8,))

opt.zero_grad()
loss = crit(model(x), y)
loss.backward()

# Grad norm BEFORE clipping
total_norm_before = 0.0
for p in model.parameters():
    if p.grad is not None:
        total_norm_before += p.grad.norm().item() ** 2
total_norm_before = total_norm_before ** 0.5

# Clip
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# Grad norm AFTER clipping
total_norm_after = 0.0
for p in model.parameters():
    if p.grad is not None:
        total_norm_after += p.grad.norm().item() ** 2
total_norm_after = total_norm_after ** 0.5

opt.step()

print(f'Gradient norm before clipping: {total_norm_before:.4f}')
print(f'Gradient norm after  clipping: {total_norm_after:.4f}  (capped at 1.0)')
print()
print('Why clip gradients?')
print('  Exploding gradients: a single large gradient causes a huge weight update,')
print('  throwing the model into a very bad region of the loss landscape.')
print('  Clipping caps the DIRECTION but preserves direction — safer than threshold.')
print('  Standard: max_norm=1.0 for Transformer training.')

In [ ]:
# ── CELL 28 · Validation loop ─────────────────────────────────────────────────
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def evaluate(
    model:     nn.Module,
    loader:    torch.utils.data.DataLoader,
    criterion: nn.Module,
    device:    torch.device,
) -> tuple:
    """
    Returns (average_loss, accuracy).
    Two mandatory lines:
      model.eval()        — disables Dropout, puts BatchNorm in inference mode
      torch.no_grad()     — disables gradient computation (saves memory + time)
    """
    model.eval()                                 # MANDATORY
    total_loss = 0.0
    correct    = 0
    total      = 0

    with torch.no_grad():                        # MANDATORY
        for x, y in loader:
            x, y   = x.to(device), y.to(device)
            logits = model(x)
            loss   = criterion(logits, y)

            total_loss += loss.item() * x.size(0)  # weight by batch size
            preds       = logits.argmax(dim=1)
            correct    += (preds == y).sum().item()
            total      += x.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    model.train()                                # restore train mode for next epoch
    return avg_loss, accuracy

# Demo
model = MLPClassifier(64, 128, 10).to(DEVICE)
X_val = torch.randn(256, 64)
Y_val = torch.randint(0, 10, (256,))
val_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_val, Y_val), batch_size=64
)
val_loss, val_acc = evaluate(model, val_loader, nn.CrossEntropyLoss(), DEVICE)
print(f'Validation loss : {val_loss:.4f}')
print(f'Validation acc  : {val_acc:.3f}  (random init, ~10% for 10 classes)')
print(f'model.training  : {model.training}  (eval() followed by train() inside evaluate())')

In [ ]:
# ── CELL 29 · Checkpointing: save and resume full training state ──────────────
import torch
import torch.nn as nn
import os, tempfile

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tmpdir = tempfile.mkdtemp()

# ── Save a full checkpoint ────────────────────────────────────────────────────
model     = MLPClassifier(64, 128, 10).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
scaler    = torch.amp.GradScaler('cuda')

# Train for one step to give optimiser real state
model.train()
x = torch.randn(8, 64, device=DEVICE)
y = torch.randint(0, 10, (8,), device=DEVICE)
with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
    loss = nn.CrossEntropyLoss()(model(x), y)
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
scheduler.step()

# Save everything
checkpoint = {
    'epoch'          : 5,
    'model'          : model.state_dict(),
    'optimizer'      : optimizer.state_dict(),
    'scheduler'      : scheduler.state_dict(),
    'scaler'         : scaler.state_dict(),
    'val_loss'       : 1.234,
    'val_acc'        : 0.567,
    'config'         : {'input_dim': 64, 'hidden_dim': 128, 'output_dim': 10},
}
ckpt_path = os.path.join(tmpdir, 'checkpoint_epoch005.pth')
torch.save(checkpoint, ckpt_path)
print(f'Checkpoint saved: {ckpt_path}')
print(f'File size       : {os.path.getsize(ckpt_path):,} bytes')

# ── Resume from checkpoint ────────────────────────────────────────────────────
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

model2     = MLPClassifier(**ckpt['config']).to(DEVICE)
optimizer2 = torch.optim.AdamW(model2.parameters(), lr=3e-4)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=100)
scaler2    = torch.amp.GradScaler('cuda')

model2.load_state_dict(ckpt['model'])
optimizer2.load_state_dict(ckpt['optimizer'])
scheduler2.load_state_dict(ckpt['scheduler'])
scaler2.load_state_dict(ckpt['scaler'])
start_epoch = ckpt['epoch'] + 1

print(f'\nResumed from epoch {ckpt["epoch"]}  (next epoch: {start_epoch})')
print(f'val_loss at save : {ckpt["val_loss"]}')
print(f'val_acc  at save : {ckpt["val_acc"]}')

# Verify model weights match
with torch.no_grad():
    diff = sum((p1-p2).abs().max().item()
               for p1,p2 in zip(model.parameters(), model2.parameters()))
print(f'\nMax weight diff between saved and loaded model: {diff:.2e}  (should be 0)')

In [ ]:
# ── CELL 30 · Best-model tracking pattern ────────────────────────────────────
import torch
import torch.nn as nn
import os, tempfile

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tmpdir  = tempfile.mkdtemp()

model     = MLPClassifier(64, 128, 10).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

# Simulate 5 epochs of training and validation
simulated_val_losses = [1.8, 1.4, 1.1, 1.3, 1.5]  # improves then worsens

best_val_loss = float('inf')
best_path     = os.path.join(tmpdir, 'best_model.pth')

for epoch, val_loss in enumerate(simulated_val_losses, 1):
    # ... training loop would be here ...

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_path)
        flag = '  ← SAVED (new best)'
    else:
        flag = ''
    print(f'Epoch {epoch}: val_loss={val_loss:.1f}  best={best_val_loss:.1f}{flag}')

# Load best model for deployment
model.load_state_dict(torch.load(best_path, weights_only=True))
print(f'\nFinal model loaded from best checkpoint (epoch 3, val_loss=1.1)')
print('Use this model for evaluation and deployment — not the last epoch!')

In [ ]:
# ── CELL 31 · Timing GPU operations correctly ─────────────────────────────────
import torch
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MLPClassifier(1024, 1024, 100).to(DEVICE)
x     = torch.randn(64, 1024, device=DEVICE)

model.eval()
with torch.no_grad():
    # Warmup
    for _ in range(5): _ = model(x)

    # WRONG: no synchronise — measures CPU dispatch time only
    t0_wrong = time.perf_counter()
    for _ in range(100): _ = model(x)
    t1_wrong = time.perf_counter()
    wrong_ms = (t1_wrong - t0_wrong) / 100 * 1000

    # CORRECT: synchronise before and after timing window
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0_right = time.perf_counter()
    for _ in range(100): _ = model(x)
    if torch.cuda.is_available(): torch.cuda.synchronize()  # wait for GPU to finish
    t1_right = time.perf_counter()
    right_ms = (t1_right - t0_right) / 100 * 1000

print(f'Forward pass timing (100 runs):')
print(f'  WITHOUT synchronise: {wrong_ms:.3f} ms  <- measures CPU dispatch, NOT GPU compute')
print(f'  WITH    synchronise: {right_ms:.3f} ms  <- true GPU execution time')
print()
if torch.cuda.is_available() and wrong_ms < right_ms * 0.5:
    print(f'  Without sync was {right_ms/wrong_ms:.0f}x faster — that is a LIE.')
    print(f'  GPU operations are asynchronous: Python returns immediately,')
    print(f'  the GPU continues working in the background.')
print('Always use torch.cuda.synchronize() when benchmarking GPU code.')

In [ ]:
# ── CELL 32 · Segment 3 review: complete training loop with everything ─────────
import torch
import torch.nn as nn
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)

# Setup
model     = MLPClassifier(128, 256, 10).to(DEVICE)
model     = torch.compile(model) if hasattr(torch, 'compile') else model
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler    = torch.amp.GradScaler('cuda')

# Synthetic dataset
X = torch.randn(1024, 128); Y = torch.randint(0, 10, (1024,))
train_ds = torch.utils.data.TensorDataset(X[:800], Y[:800])
val_ds   = torch.utils.data.TensorDataset(X[800:], Y[800:])
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=128)

USE_AMP = torch.cuda.is_available()
print(f'Training on {DEVICE}, AMP={USE_AMP}\n')
print(f'  {"Epoch":<6} {"tr_loss":>8} {"val_loss":>10} {"val_acc":>10} {"LR":>12} {"time":>8}')
print('  ' + '-'*56)

best_val_loss = float('inf')
for epoch in range(1, 6):
    t0 = time.perf_counter()

    # ── Training ────────────────────────────────────────────────────────────
    model.train()
    tr_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=USE_AMP):
            logits = model(bx)
            loss   = criterion(logits, by)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        tr_loss += loss.item()
    scheduler.step()

    # ── Validation ──────────────────────────────────────────────────────────
    vl, va = evaluate(model, val_loader, criterion, DEVICE)

    elapsed = time.perf_counter() - t0
    lr = optimizer.param_groups[0]['lr']
    print(f'  {epoch:<6} {tr_loss/len(train_loader):>8.4f} {vl:>10.4f} {va:>10.3f} {lr:>12.6f} {elapsed:>6.2f}s')

print('\nTraining complete!')

---
## Segment 4 · DataLoader — Feeding the Model Efficiently
### Slides 24–30

In [ ]:
# ── CELL 33 · Writing a custom Dataset from scratch ───────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
import os, tempfile
import numpy as np

# ── Minimal Dataset contract ──────────────────────────────────────────────────
class SyntheticDataset(Dataset):
    """
    The simplest possible Dataset.
    __len__     : total number of samples
    __getitem__ : return one (input, label) pair for a given index
    """
    def __init__(self, n_samples: int, input_dim: int, n_classes: int):
        self.X = torch.randn(n_samples, input_dim)
        self.Y = torch.randint(0, n_classes, (n_samples,))

    def __len__(self) -> int:
        return len(self.X)            # total samples

    def __getitem__(self, idx: int):
        return self.X[idx], self.Y[idx]  # one sample

ds = SyntheticDataset(n_samples=1000, input_dim=32, n_classes=5)
print(f'Dataset length      : {len(ds)}')
x0, y0 = ds[0]
print(f'ds[0] -> x={x0.shape}, y={y0}')
x5, y5 = ds[42]
print(f'ds[42] -> x={x5.shape}, y={y5}')

# ── Dataset that reads from disk (lazy loading) ───────────────────────────────
class CSVDataset(Dataset):
    """
    Demonstrates best practice:
    - __init__: only store file paths and metadata (fast)
    - __getitem__: actually load and process each sample (lazy)
    """
    def __init__(self, csv_path: str):
        data = np.loadtxt(csv_path, delimiter=',')
        self.X = torch.tensor(data[:, :-1], dtype=torch.float32)
        self.Y = torch.tensor(data[:,  -1], dtype=torch.long)

    def __len__(self): return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# Create a tiny CSV and test
tmpdir  = tempfile.mkdtemp()
csvpath = os.path.join(tmpdir, 'data.csv')
np.savetxt(csvpath, np.random.randn(100, 5), delimiter=',')
csv_ds  = CSVDataset(csvpath)
print(f'\nCSVDataset length   : {len(csv_ds)}')
print(f'Feature shape       : {csv_ds[0][0].shape}')

In [ ]:
# ── CELL 34 · Custom Dataset for image classification ─────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os, tempfile
from PIL import Image
import numpy as np

class ImageFolderDataset(Dataset):
    """
    Reads images from a list of file paths.
    Demonstrates:
    - PIL image loading and conversion
    - applying torchvision transforms in __getitem__
    - lazy loading (nothing loaded in __init__)
    """
    def __init__(self, paths: list, labels: list, transform=None):
        assert len(paths) == len(labels)
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx: int):
        img   = Image.open(self.paths[idx]).convert('RGB')  # lazy: load on demand
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)   # returns float tensor
        return img, torch.tensor(label, dtype=torch.long)

# Create synthetic 'images' saved as PNG files
tmpdir = tempfile.mkdtemp()
paths, labels = [], []
for i in range(20):
    arr = (np.random.rand(32, 32, 3) * 255).astype(np.uint8)
    p   = os.path.join(tmpdir, f'img_{i:03d}.png')
    Image.fromarray(arr).save(p)
    paths.append(p)
    labels.append(i % 5)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

img_ds = ImageFolderDataset(paths, labels, transform=transform)
img, lbl = img_ds[0]
print(f'ImageFolderDataset:')
print(f'  Length          : {len(img_ds)}')
print(f'  Image tensor    : shape={img.shape}, dtype={img.dtype}')
print(f'  Pixel range     : [{img.min():.2f}, {img.max():.2f}]  (normalised)')
print(f'  Label           : {lbl.item()}')

In [ ]:
# ── CELL 35 · DataLoader: every parameter explained ──────────────────────────
import torch
from torch.utils.data import DataLoader

ds = SyntheticDataset(n_samples=500, input_dim=32, n_classes=5)

# ── Minimal DataLoader ────────────────────────────────────────────────────────
loader_minimal = DataLoader(ds, batch_size=32)

# ── Production DataLoader ─────────────────────────────────────────────────────
loader_prod = DataLoader(
    ds,
    batch_size          = 64,
    shuffle             = True,        # RandomSampler — different order each epoch
    num_workers         = 2,           # parallel workers (use 4-8 in practice)
    pin_memory          = True,        # page-lock CPU tensors for faster GPU transfer
    drop_last           = True,        # discard partial last batch
    persistent_workers  = True,        # keep workers alive between epochs
    prefetch_factor     = 2,           # batches to pre-load per worker
)

# Inspect a batch
bx, by = next(iter(loader_prod))
print(f'Batch shape  : x={bx.shape}, y={by.shape}')
print(f'pin_memory   : {bx.is_pinned()}')

# Compare batch counts (drop_last removes final partial batch)
n_minimal = sum(1 for _ in loader_minimal)
n_prod    = sum(1 for _ in loader_prod)
print(f'\nBatches per epoch:')
print(f'  minimal (bs=32, drop_last=False): {n_minimal}  = ceil({len(ds)}/32)')
print(f'  prod    (bs=64, drop_last=True) : {n_prod}   = floor({len(ds)}/64)')

# Demonstrate shuffle
loader_noshuffle = DataLoader(ds, batch_size=4, shuffle=False)
loader_shuffle   = DataLoader(ds, batch_size=4, shuffle=True)
_, y_no = next(iter(loader_noshuffle))
_, y_sh = next(iter(loader_shuffle))
print(f'\nFirst batch labels (no shuffle) : {y_no.tolist()}')
print(f'First batch labels (shuffled)   : {y_sh.tolist()}  <- different each run')

In [ ]:
# ── CELL 36 · Custom collate_fn for variable-length sequences ─────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class VarLenDataset(Dataset):
    """Each sample has a different sequence length (realistic NLP scenario)."""
    def __init__(self, n=50, vocab=1000, max_len=128):
        self.n = n
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self): return self.n

    def __getitem__(self, idx):
        length = torch.randint(5, self.max_len, (1,)).item()
        seq    = torch.randint(1, self.vocab, (length,))   # token IDs (no padding)
        label  = torch.tensor(idx % 2)                    # binary label
        return seq, label

def pad_collate(batch):
    """
    Custom collate function:
    - Pads sequences to the max length in this batch
    - Returns: (padded_seqs, lengths, labels)
    """
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    padded  = pad_sequence(seqs, batch_first=True, padding_value=0)  # (B, T_max)
    labels  = torch.stack(labels)
    return padded, lengths, labels

loader = DataLoader(VarLenDataset(n=32), batch_size=8, collate_fn=pad_collate)
padded, lengths, labels = next(iter(loader))

print(f'Variable-length batch:')
print(f'  padded shape  : {padded.shape}   (B, T_max)')
print(f'  lengths       : {lengths.tolist()}')
print(f'  labels        : {labels.tolist()}')
print(f'  padding value : {padded[0, lengths[0]:lengths[0]+3].tolist()}  (zeros after real tokens)')
print()
print('Why custom collate_fn?')
print('  Default collate_fn requires all tensors to have the SAME shape.')
print('  pad_sequence aligns variable-length sequences by padding to max length.')
print('  The lengths tensor tells the model where real tokens end.')

In [ ]:
# ── CELL 37 · Image transforms: training vs validation ───────────────────────
import torch
import torchvision.transforms as T
import torchvision.transforms.v2 as T2
from PIL import Image
import numpy as np

# Create a synthetic image for demonstration
pil_img = Image.fromarray(
    (np.random.rand(64, 64, 3) * 255).astype(np.uint8)
)

# ── Training transforms (add augmentation) ───────────────────────────────────
train_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.5, 1.0)),  # random crop + resize
    T.RandomHorizontalFlip(p=0.5),               # 50% mirror
    T.ColorJitter(                               # random colour distortion
        brightness=0.3, contrast=0.3,
        saturation=0.3, hue=0.1),
    T.ToTensor(),                                # PIL -> tensor [0,1], CHW
    T.Normalize(mean=[0.485, 0.456, 0.406],      # ImageNet stats
                std =[0.229, 0.224, 0.225]),
])

# ── Validation transforms (deterministic only) ────────────────────────────────
val_transform = T.Compose([
    T.Resize(40),            # resize shorter edge to 40
    T.CenterCrop(32),        # crop centre 32x32
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

# Apply and compare
t_train = train_transform(pil_img)
t_val   = val_transform(pil_img)

print('Transform outputs:')
print(f'  Training   : shape={t_train.shape}, range=[{t_train.min():.2f}, {t_train.max():.2f}]')
print(f'  Validation : shape={t_val.shape},   range=[{t_val.min():.2f}, {t_val.max():.2f}]')

# Show that train transforms are random
t1 = train_transform(pil_img)
t2 = train_transform(pil_img)
print(f'\nTrain transform produces different results each time:')
print(f'  max diff between two runs: {(t1-t2).abs().max():.4f}  (should be >0)')

# Val transforms are deterministic
v1 = val_transform(pil_img)
v2 = val_transform(pil_img)
print(f'Val transform is deterministic:')
print(f'  max diff between two runs: {(v1-v2).abs().max():.4f}  (should be 0)')

print('\nKey rule: NEVER add random augmentation to validation or test transforms!')

In [ ]:
# ── CELL 38 · Computing dataset normalisation statistics ──────────────────────
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

# When NOT fine-tuning a pretrained model, compute stats from YOUR dataset
class ImageDatasetForStats(torch.utils.data.Dataset):
    """Returns images as [0,1] float tensors without normalisation."""
    def __init__(self, n=500):
        # Simulate non-ImageNet images (e.g. medical, satellite)
        self.imgs = torch.rand(n, 3, 32, 32) * 0.6 + 0.2  # not ImageNet distribution
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.imgs[i], 0

loader  = DataLoader(ImageDatasetForStats(500), batch_size=100)
channel_sum   = torch.zeros(3)
channel_sum_sq = torch.zeros(3)
n_pixels = 0

for imgs, _ in loader:
    # imgs: (B, C, H, W)
    B, C, H, W   = imgs.shape
    channel_sum   += imgs.sum(dim=[0, 2, 3])     # sum over B,H,W
    channel_sum_sq += (imgs**2).sum(dim=[0, 2, 3])
    n_pixels      += B * H * W

mean = channel_sum   / n_pixels
std  = (channel_sum_sq / n_pixels - mean**2).sqrt()

print('Dataset normalisation statistics:')
print(f'  Computed mean : {mean.tolist()}')
print(f'  Computed std  : {std.tolist()}')
print(f'\n  ImageNet mean : [0.485, 0.456, 0.406]')
print(f'  ImageNet std  : [0.229, 0.224, 0.225]')
print()
print('Use ImageNet stats ONLY when fine-tuning ImageNet-pretrained models.')
print('For your own datasets: compute these stats from YOUR training set.')

In [ ]:
# ── CELL 39 · Using torchvision built-in datasets ─────────────────────────────
import torch
import torchvision
import torchvision.transforms as T
import os

# CIFAR-10 — downloads automatically if not present
data_dir = os.path.join(os.environ.get('HOME', '/tmp'), 'datasets')

train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010)),
])
val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010)),
])

print('Loading CIFAR-10 (downloading if needed)...')
try:
    train_ds = torchvision.datasets.CIFAR10(
        root=data_dir, train=True, download=True, transform=train_transform)
    val_ds = torchvision.datasets.CIFAR10(
        root=data_dir, train=False, download=True, transform=val_transform)

    print(f'  Training set  : {len(train_ds):,} samples')
    print(f'  Validation set: {len(val_ds):,} samples')
    img, lbl = train_ds[0]
    print(f'  Sample: image={img.shape}, label={lbl}')
    print(f'  Classes: {train_ds.classes}')

    # Create DataLoaders
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=128, shuffle=True, num_workers=2,
        pin_memory=True, persistent_workers=True, drop_last=True)
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

    bx, by = next(iter(train_loader))
    print(f'\n  Train batch: x={bx.shape}, y={by.shape}')
    print(f'  Batches per epoch: {len(train_loader)}')

except Exception as e:
    print(f'  Download skipped: {e}')
    print('  Run with internet access to download CIFAR-10 automatically.')

In [ ]:
# ── CELL 40 · DataLoader bottleneck diagnostic ────────────────────────────────
import torch
import time
from torch.utils.data import DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SlowDataset(torch.utils.data.Dataset):
    """Simulates slow disk I/O with a sleep in __getitem__."""
    def __init__(self, n=512, slow=False):
        self.n = n; self.slow = slow
    def __len__(self): return self.n
    def __getitem__(self, i):
        if self.slow:
            import time
            time.sleep(0.001)   # simulate 1ms disk read
        return torch.randn(128), torch.randint(0, 4, (1,)).squeeze()

class DummyDataset(torch.utils.data.Dataset):
    """Instant: returns pre-generated random tensors. Used to isolate GPU bottleneck."""
    def __init__(self, n=512): self.X = torch.randn(n,128); self.Y = torch.randint(0,4,(n,))
    def __len__(self): return len(self.Y)
    def __getitem__(self, i): return self.X[i], self.Y[i]

model_bench = MLPClassifier(128, 256, 4).to(DEVICE).eval()

def time_pipeline(dataset, nw, label, n_batches=20):
    loader = DataLoader(dataset, batch_size=32, num_workers=nw,
                        pin_memory=(DEVICE.type=='cuda'),
                        persistent_workers=(nw>0),
                        prefetch_factor=2 if nw>0 else None)
    t0 = time.perf_counter()
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= n_batches: break
            x = x.to(DEVICE)
            _ = model_bench(x)
            if DEVICE.type=='cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t0) / min(n_batches, len(loader)) * 1000

configs = [
    (DummyDataset(512), 0, 'Dummy dataset, nw=0 (GPU baseline)'),
    (SlowDataset(512, slow=False), 0, 'Real dataset, nw=0 (single-process)'),
    (SlowDataset(512, slow=False), 2, 'Real dataset, nw=2 (parallel)'),
    (SlowDataset(512, slow=True),  0, 'SLOW dataset, nw=0 (1ms/sample)'),
    (SlowDataset(512, slow=True),  4, 'SLOW dataset, nw=4 (parallel fix)'),
]

print(f'  {"Config":<43}  {"ms/batch":>10}')
print('  ' + '-'*56)
for ds, nw, label in configs:
    ms = time_pipeline(ds, nw, label)
    print(f'  {label:<43}  {ms:>10.2f}')

print('\nConclusion:')
print('  Dummy dataset = pure GPU compute time (lower bound)')
print('  If real dataset >> dummy: DataLoader is the bottleneck')
print('  Fix: increase num_workers until real dataset approaches dummy time')

In [ ]:
# ── CELL 41 · Samplers: SequentialSampler, RandomSampler, WeightedRandomSampler
import torch
from torch.utils.data import SequentialSampler, RandomSampler, WeightedRandomSampler

ds_small = SyntheticDataset(20, 4, 3)

# ── SequentialSampler ─────────────────────────────────────────────────────────
seq = list(SequentialSampler(ds_small))
print(f'SequentialSampler  : {seq}')

# ── RandomSampler ─────────────────────────────────────────────────────────────
rand = list(RandomSampler(ds_small))
print(f'RandomSampler      : {rand}')

# ── WeightedRandomSampler: over-sample rare classes ───────────────────────────
# Simulate imbalanced dataset: 80% class 0, 10% class 1, 10% class 2
class_counts = [16, 2, 2]
labels       = [0]*16 + [1]*2 + [2]*2
weights_per_class = [1.0/c for c in class_counts]     # inverse frequency
sample_weights    = [weights_per_class[l] for l in labels]
sample_weights    = torch.tensor(sample_weights)

weighted_sampler  = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=20,
    replacement=True,
)
weighted_indices = list(weighted_sampler)
sampled_labels   = [labels[i] for i in weighted_indices]
from collections import Counter
print(f'\nWeightedRandomSampler (20 samples from imbalanced dataset):')
print(f'  Original class counts   : {Counter(labels)}')
print(f'  Sampled  class counts   : {Counter(sampled_labels)}')
print(f'  Class 1 and 2 now appear more often — class imbalance corrected!')

In [ ]:
# ── CELL 42 · pin_memory and non_blocking transfer demonstration ──────────────
import torch
import time

if not torch.cuda.is_available():
    print('GPU required for this cell. Showing code pattern only.')
    print("""\n# Pattern to use in your training loop:
for x, y in train_loader:
    # pin_memory=True in DataLoader + non_blocking=True here
    # enables async DMA: transfer happens while GPU computes previous batch
    x = x.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)
    # ... rest of training step
""")
else:
    SIZE = int(1e8)   # 100M float32 = 400MB
    device = torch.device('cuda')

    cpu_pageable = torch.randn(SIZE)
    cpu_pinned   = torch.randn(SIZE).pin_memory()

    def bw(src, nb, n=10):
        gpu = torch.empty(SIZE, device=device)
        for _ in range(3): gpu.copy_(src, non_blocking=nb)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n): gpu.copy_(src, non_blocking=nb)
        torch.cuda.synchronize()
        return (SIZE * 4 * n) / (time.perf_counter() - t0) / 1e9  # GB/s

    print(f'PCIe Transfer Bandwidth (400 MB tensor, 10 runs):')
    print(f'  pageable, non_blocking=False : {bw(cpu_pageable, False):>6.2f} GB/s')
    print(f'  pinned,   non_blocking=False : {bw(cpu_pinned,   False):>6.2f} GB/s')
    print(f'  pinned,   non_blocking=True  : {bw(cpu_pinned,   True ):>6.2f} GB/s  (async DMA)')
    print(f'\nPCIe Gen4 x16 theoretical peak: ~32 GB/s')
    print(f'Always use pin_memory=True in DataLoader + non_blocking=True in .to()!')

In [ ]:
# ── CELL 43 · Segment 4 review: training CNN on CIFAR-10 ─────────────────────
# Requires Cell 21 (SimpleCNN) and Cell 39 (CIFAR-10 download) to have been run.
# If CIFAR-10 is not available, uses synthetic data automatically.
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Check if CIFAR-10 loaders from Cell 39 are available
try:
    _ = train_loader  # from Cell 39
    USE_CIFAR = True
    print('Using CIFAR-10 DataLoaders from Cell 39.')
except NameError:
    USE_CIFAR = False
    print('CIFAR-10 not loaded (Cell 39 not run or download failed).')
    print('Using 3x32x32 synthetic images instead.')

# Check if SimpleCNN is defined (from Cell 21)
try:
    _ = SimpleCNN
except NameError:
    # Define inline if Cell 21 was not run
    class SimpleCNN(nn.Module):
        def __init__(self, num_classes=10):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            )
            self.classifier = nn.Sequential(
                nn.Flatten(), nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, num_classes)
            )
        def forward(self, x): return self.classifier(self.features(x))
    print('SimpleCNN defined inline (Cell 21 was not run).')

if not USE_CIFAR:
    # Synthetic 3x32x32 images
    synth_ds = torch.utils.data.TensorDataset(
        torch.randn(512, 3, 32, 32), torch.randint(0, 10, (512,))
    )
    train_loader_use = torch.utils.data.DataLoader(synth_ds, batch_size=64, shuffle=True)
    val_loader_use   = torch.utils.data.DataLoader(synth_ds, batch_size=128)
else:
    train_loader_use = train_loader
    val_loader_use   = val_loader

model_c = SimpleCNN(num_classes=10).to(DEVICE)
opt_c   = torch.optim.AdamW(model_c.parameters(), lr=3e-4, weight_decay=0.01)
crit_c  = nn.CrossEntropyLoss()

print(f'\nSimpleCNN on {"CIFAR-10" if USE_CIFAR else "synthetic 3x32x32"}, device={DEVICE}')
print(f'{"Epoch":<6}  {"Loss":>8}  {"Acc":>8}')

for epoch in range(1, 4):
    model_c.train()
    tr_loss = 0.0
    for bx, by in train_loader_use:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        opt_c.zero_grad(set_to_none=True)
        loss = crit_c(model_c(bx), by)
        loss.backward()
        opt_c.step()
        tr_loss += loss.item()

    model_c.eval()
    correct = total_n = 0
    with torch.no_grad():
        for bx, by in val_loader_use:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            preds    = model_c(bx).argmax(1)
            correct += (preds == by).sum().item()
            total_n += by.size(0)
    print(f'{epoch:<6}  {tr_loss/len(train_loader_use):>8.4f}  {correct/total_n:>8.4f}')


---
## Segment 5 · First Steps in Performance: AMP, torch.compile & Key Practices
### Slides 31–38

In [ ]:
# ── CELL 44 · AMP Part 1: What BF16 is and why it matters ────────────────────
import torch

print('Floating Point Format Comparison:')
print(f'  {"Format":<12} {"Total bits":>10} {"Exponent":>9} {"Mantissa":>9} {"Dynamic range":>15}')
print('  ' + '-'*57)
formats = [
    ('FP32',     32, 8,  23, '~1e-38 to ~3e38'),
    ('FP16',     16, 5,  10, '~6e-5  to ~65504'),
    ('BF16',     16, 8,   7, '~1e-38 to ~3e38'),
]
for name, tot, exp, man, rng in formats:
    print(f'  {name:<12} {tot:>10} {exp:>9} {man:>9} {rng:>15}')

print('\nKey insight:')
print('  BF16 has the same exponent as FP32 (same dynamic range)')
print('  but only 7 mantissa bits (less precision than FP16).')
print('  Result: BF16 is safe for training without loss scaling.')
print('  FP16 has only 5 exponent bits — large gradients can overflow to Inf!')

# Demonstrate overflow risk in FP16
large_val = torch.tensor(65536.0)   # just above FP16 max (~65504)
fp16_val  = large_val.to(torch.float16)
bf16_val  = large_val.to(torch.bfloat16)
print(f'\nValue: {large_val.item()}')
print(f'  As FP16  : {fp16_val.item()}   <- OVERFLOW to infinity!')
print(f'  As BF16  : {bf16_val.item()}    <- OK (wider exponent)')
print(f'  As FP32  : {large_val.item()}  <- OK')

# Memory saving
n_params = 7e9  # 7B parameter model
print(f'\n7B parameter model memory:')
print(f'  FP32 : {n_params*4/1e9:.0f} GB')
print(f'  BF16 : {n_params*2/1e9:.0f} GB  <- 2x less VRAM')

In [ ]:
# ── CELL 45 · AMP Part 2: Implementation ─────────────────────────────────────
import torch
import torch.nn as nn
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_GPU = DEVICE.type == 'cuda'

model = MLPClassifier(512, 1024, 100).to(DEVICE).train()
opt   = torch.optim.AdamW(model.parameters(), lr=3e-4)
crit  = nn.CrossEntropyLoss()
x     = torch.randn(128, 512, device=DEVICE)
y     = torch.randint(0, 100, (128,), device=DEVICE)

# ── Pattern 1: FP32 (baseline) ───────────────────────────────────────────────
def step_fp32():
    opt.zero_grad(set_to_none=True)
    loss = crit(model(x), y)
    loss.backward()
    opt.step()
    return loss

# ── Pattern 2: BF16 AMP (recommended for A100/H100) ──────────────────────────
def step_amp_bf16():
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=USE_GPU):
        logits = model(x)
        loss   = crit(logits, y)
    loss.backward()    # no GradScaler needed for BF16
    opt.step()
    return loss

# ── Pattern 3: FP16 AMP with GradScaler (for V100 and older GPUs) ────────────
scaler = torch.amp.GradScaler('cuda')
def step_amp_fp16():
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', dtype=torch.float16, enabled=USE_GPU):
        logits = model(x)
        loss   = crit(logits, y)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()
    return loss

# Benchmark
def bench(fn, n=50, label=''):
    for _ in range(5): fn()  # warmup
    if USE_GPU: torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n): fn()
    if USE_GPU: torch.cuda.synchronize()
    return (time.perf_counter()-t0)/n*1000

t_fp32  = bench(step_fp32,     label='FP32')
t_bf16  = bench(step_amp_bf16, label='BF16 AMP')
t_fp16  = bench(step_amp_fp16, label='FP16 AMP')

print('AMP Benchmark (forward + backward, batch=128):')
print(f'  FP32 (baseline)  : {t_fp32:.2f} ms')
print(f'  BF16 AMP         : {t_bf16:.2f} ms  ({t_fp32/t_bf16:.2f}x speedup)')
print(f'  FP16 AMP+scaler  : {t_fp16:.2f} ms  ({t_fp32/t_fp16:.2f}x speedup)')
if USE_GPU:
    print(f'\n  GPU VRAM used: {torch.cuda.memory_allocated()/1e6:.0f} MB')

In [ ]:
# ── CELL 46 · torch.compile: basic usage and compilation warmup ───────────────
import torch
import torch.nn as nn
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Eager (baseline) ─────────────────────────────────────────────────────────
model_eager = MLPClassifier(512, 1024, 100).to(DEVICE).eval()
x = torch.randn(64, 512, device=DEVICE)

with torch.no_grad():
    for _ in range(5): model_eager(x)  # warmup
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(100): model_eager(x)
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    t_eager = (time.perf_counter()-t0)/100*1000

# ── Compiled ─────────────────────────────────────────────────────────────────
import torch._dynamo
torch._dynamo.reset()

# ONE LINE CHANGE — everything else stays the same
model_compiled = torch.compile(
    MLPClassifier(512, 1024, 100).to(DEVICE).eval(),
    mode='default',      # balanced: good speedup, moderate compile time
)

# IMPORTANT: first N calls trigger compilation — they are SLOWER
print('Compiling (first 5 calls — these will be slow)...')
compilation_times = []
with torch.no_grad():
    for i in range(5):
        t0 = time.perf_counter()
        model_compiled(x)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        compilation_times.append((time.perf_counter()-t0)*1000)
print(f'  Call 1: {compilation_times[0]:.0f} ms  (compilation happening)')
print(f'  Call 3: {compilation_times[2]:.0f} ms')
print(f'  Call 5: {compilation_times[4]:.0f} ms  (compilation largely done)')

# Now benchmark steady-state performance
with torch.no_grad():
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(100): model_compiled(x)
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    t_compiled = (time.perf_counter()-t0)/100*1000

print(f'\nSteady-state benchmark (100 forward passes):')
print(f'  Eager    : {t_eager:.3f} ms')
print(f'  Compiled : {t_compiled:.3f} ms  ({t_eager/t_compiled:.2f}x speedup)')
print()
print('Rule: do not benchmark the first 5-10 calls — they include compilation time.')

In [ ]:
# ── CELL 47 · torch.compile modes ─────────────────────────────────────────────
import torch
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = torch.randn(64, 512, device=DEVICE)

modes = [
    ('default',         'balanced compile time / speedup, best starting point'),
    ('reduce-overhead', 'enables CUDA Graphs, needs fixed tensor shapes'),
]
# ('max-autotune', 'tries many kernel implementations, best runtime, slow compile')
# Excluded for time: max-autotune takes 5-10 minutes

print(f'  {"Mode":<20}  {"ms/step":>8}  Description')
print('  ' + '-'*70)

# Eager baseline
m_base = MLPClassifier(512, 1024, 100).to(DEVICE).eval()
with torch.no_grad():
    for _ in range(5): m_base(x)
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(50): m_base(x)
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    t_base = (time.perf_counter()-t0)/50*1000
print(f'  {"eager (baseline)":<20}  {t_base:>8.3f}  no compilation')

for mode, desc in modes:
    import torch._dynamo; torch._dynamo.reset()
    m = torch.compile(MLPClassifier(512,1024,100).to(DEVICE).eval(), mode=mode)
    with torch.no_grad():
        for _ in range(8): m(x)    # warmup through compilation
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(50): m(x)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        t_m = (time.perf_counter()-t0)/50*1000
    speedup = t_base/t_m
    print(f'  {mode:<20}  {t_m:>8.3f}  {speedup:.2f}x — {desc}')

In [ ]:
# ── CELL 48 · TF32: free speed on Ampere+ GPUs ────────────────────────────────
import torch
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('TF32 (TensorFloat-32) Status:')
print(f'  torch.backends.cuda.matmul.allow_tf32 : {torch.backends.cuda.matmul.allow_tf32}')
print(f'  torch.backends.cudnn.allow_tf32        : {torch.backends.cudnn.allow_tf32}')
print()
print('TF32 is enabled by default on Ampere (A100) and newer GPUs.')
print('It uses the 8-bit exponent of FP32 but only 10 bits of mantissa')
print('(vs 23 bits for FP32). matmul runs at ~3x FP32 speed on Tensor Cores.')
print('Accuracy impact is negligible for nearly all AI models.')

if not torch.cuda.is_available():
    print('\nGPU not available — skipping TF32 vs FP32 matmul benchmark.')
    print('On an A100: TF32 matmul is ~3x faster than true FP32.')
else:
    N = 4096
    A = torch.randn(N, N, device=DEVICE)
    B = torch.randn(N, N, device=DEVICE)

    def bench_matmul(allow_tf32, n=20):
        torch.backends.cuda.matmul.allow_tf32 = allow_tf32
        for _ in range(3): C = A @ B  # warmup
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n): C = A @ B
        torch.cuda.synchronize()
        return (time.perf_counter()-t0)/n*1000

    t_tf32 = bench_matmul(True)
    t_fp32 = bench_matmul(False)
    torch.backends.cuda.matmul.allow_tf32 = True  # restore default

    print(f'\n  {N}x{N} matmul benchmark:')
    print(f'  TF32 (default) : {t_tf32:.2f} ms')
    print(f'  FP32 (disabled): {t_fp32:.2f} ms')
    print(f'  TF32 speedup   : {t_fp32/t_tf32:.2f}x')
    print()
    print('TF32 is already enabled — you get this speedup for free!')
    print('Only disable TF32 for numerical reproducibility tests.')

In [ ]:
# ── CELL 49 · Six common mistakes: live demonstrations ───────────────────────
import torch
import torch.nn as nn
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('SIX COMMON PYTORCH PERFORMANCE MISTAKES')
print('=' * 52)

# ── Mistake 1: model.train() / eval() ────────────────────────────────────────
print('\nMistake 1: Forgetting model.eval() before validation')
m = nn.Sequential(nn.Linear(4,8), nn.Dropout(0.9), nn.Linear(8,2)).to(DEVICE)
x = torch.randn(10, 4, device=DEVICE)
with torch.no_grad():
    m.train(); out_train = m(x).std().item()  # Dropout active
    m.eval();  out_eval  = m(x).std().item()  # Dropout disabled
    m.eval();  out_eval2 = m(x).std().item()  # Same result
print(f'  train mode std: {out_train:.4f}  (random — Dropout kills 90% of neurons)')
print(f'  eval  mode std: {out_eval:.4f}  (deterministic)')
print(f'  eval  mode run2:{out_eval2:.4f}  (same — no randomness)')

# ── Mistake 2: Forgetting zero_grad ──────────────────────────────────────────
print('\nMistake 2: Forgetting zero_grad — gradients accumulate')
w = nn.Linear(4, 2)
crit = nn.MSELoss()
grad_norms = []
for _ in range(4):
    loss = crit(w(torch.randn(8,4)), torch.randn(8,2))
    loss.backward()   # NO zero_grad!
    grad_norms.append(w.weight.grad.norm().item())
print(f'  Grad norm each step (no zero_grad): {[f"{n:.3f}" for n in grad_norms]}')
print(f'  Growing! Step 4 grad = {grad_norms[-1]/grad_norms[0]:.1f}x step 1')

# ── Mistake 3: num_workers=0 ──────────────────────────────────────────────────
print('\nMistake 3: num_workers=0 (not shown in detail — see Cell 40)')
print('  Symptom: GPU SM% oscillates between 0 and 90+ percent')
print('  Fix    : set num_workers=4 (per GPU)')

# ── Mistake 4: Wrong device ───────────────────────────────────────────────────
print('\nMistake 4: Data on wrong device')
m_gpu = nn.Linear(4,2).to(DEVICE)
try:
    _ = m_gpu(torch.randn(3, 4))   # CPU data, GPU model
except RuntimeError as e:
    print(f'  Error: {str(e)[:65]}')

# ── Mistake 5: Calling .forward() directly ───────────────────────────────────
print('\nMistake 5: model.forward(x) bypasses hooks (see Cell 23 for full demo)')

# ── Mistake 6: Timing without synchronise ─────────────────────────────────────
print('\nMistake 6: Timing without synchronise (see Cell 31 for full demo)')
if DEVICE.type=='cuda':
    m2 = MLPClassifier(512,1024,10).to(DEVICE).eval()
    x2 = torch.randn(64, 512, device=DEVICE)
    with torch.no_grad():
        for _ in range(5): m2(x2)
        t0 = time.perf_counter(); _ = m2(x2); t_wrong = (time.perf_counter()-t0)*1000
        torch.cuda.synchronize()
        t0 = time.perf_counter(); _ = m2(x2); torch.cuda.synchronize(); t_right = (time.perf_counter()-t0)*1000
    print(f'  Without sync : {t_wrong:.3f} ms  (LIE — measures CPU dispatch)')
    print(f'  With    sync : {t_right:.3f} ms  (TRUTH — measures GPU execution)')

In [ ]:
# ── CELL 50 · set_to_none=True and other small wins ──────────────────────────
import torch
import torch.nn as nn
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLPClassifier(512, 1024, 100).to(DEVICE).train()
opt   = torch.optim.AdamW(model.parameters(), lr=3e-4)
crit  = nn.CrossEntropyLoss()
x     = torch.randn(128, 512, device=DEVICE)
y     = torch.randint(0, 100, (128,), device=DEVICE)

def bench_step(zero_none, n=100):
    for _ in range(5):
        opt.zero_grad(set_to_none=zero_none)
        crit(model(x), y).backward()
        opt.step()
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        opt.zero_grad(set_to_none=zero_none)
        crit(model(x), y).backward()
        opt.step()
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    return (time.perf_counter()-t0)/n*1000

t_false = bench_step(zero_none=False)
t_true  = bench_step(zero_none=True)

print(f'zero_grad(set_to_none=False): {t_false:.3f} ms  (fills with zeros)')
print(f'zero_grad(set_to_none=True) : {t_true:.3f} ms  (sets to None, allows dealloc)')
print(f'Speedup: {t_false/t_true:.3f}x  (small but free)')

print('\nOther small wins summary:')
smalls = [
    ('zero_grad(set_to_none=True)', 'Frees gradient memory instead of filling zeros'),
    ('DataLoader drop_last=True',   'Avoids BatchNorm instability on tiny last batch'),
    ('DataLoader persistent_workers=True', '2-5s faster epoch start (no re-spawn)'),
    ('torch.backends.cuda.matmul.allow_tf32=True', 'Default; ~3x matmul on A100 (free!)'),
    ('model.to(memory_format=torch.channels_last)', '~10% CNN speedup on Tensor Cores'),
    ('torch.set_float32_matmul_precision("high")', 'Same as TF32 for matmul; no-op if already set'),
]
for name, desc in smalls:
    print(f'  {name:<46} -> {desc}')

In [ ]:
# ── CELL 51 · Demonstrating torch.compile with AMP together ──────────────────
import torch
import torch.nn as nn
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_GPU = DEVICE.type == 'cuda'

x = torch.randn(128, 512, device=DEVICE)
y = torch.randint(0, 100, (128,), device=DEVICE)

def make_model():
    return MLPClassifier(512, 1024, 100).to(DEVICE).train()

def timed_steps(model, n_steps=80, use_amp=False, label=''):
    opt    = torch.optim.AdamW(model.parameters(), lr=3e-4)
    crit   = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and USE_GPU))

    for _ in range(5):
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=(use_amp and USE_GPU)):
            loss = crit(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()

    if USE_GPU: torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_steps):
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=(use_amp and USE_GPU)):
            loss = crit(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
    if USE_GPU: torch.cuda.synchronize()
    return (time.perf_counter()-t0)/n_steps*1000

configs = [
    (make_model(),                                False, 'Eager FP32'),
    (make_model(),                                True,  'Eager + BF16 AMP'),
]
if hasattr(torch, 'compile'):
    import torch._dynamo
    torch._dynamo.reset()
    cm = torch.compile(make_model(), mode='default')
    configs.append((cm, False, 'Compiled + FP32'))
    torch._dynamo.reset()
    cam = torch.compile(make_model(), mode='default')
    configs.append((cam, True,  'Compiled + BF16 AMP  <- recommended'))

baseline = None
print(f'  {"Configuration":<36} {"ms/step":>8}  {"Speedup":>8}')
print('  ' + '-'*55)
for model_cfg, amp, label in configs:
    ms = timed_steps(model_cfg, use_amp=amp)
    if baseline is None: baseline = ms
    print(f'  {label:<36} {ms:>8.3f}  {baseline/ms:>7.2f}x')

In [ ]:
# ── CELL 52 · Performance quick-start checklist as runnable verification ──────
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MLPClassifier(64, 128, 10).to(DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

print('PERFORMANCE QUICK-START CHECKLIST')
print('=' * 50)

checks = []

# 1. BF16 available
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
checks.append(('BF16 AMP available', bf16_ok))

# 2. Flash SDPA enabled
checks.append(('FlashAttention/SDPA enabled', torch.backends.cuda.flash_sdp_enabled()))

# 3. torch.compile available
checks.append(('torch.compile available', int(torch.__version__.split('.')[0]) >= 2))

# 4. TF32 matmul
checks.append(('TF32 matmul enabled (free on A100)', torch.backends.cuda.matmul.allow_tf32))

# 5. TF32 cuDNN
checks.append(('TF32 cuDNN enabled', torch.backends.cudnn.allow_tf32))

# 6. GPU available
checks.append(('CUDA GPU available', torch.cuda.is_available()))

# 7. cudnn benchmark (off for reproducibility)
checks.append(('cuDNN deterministic (benchmark=False)', not torch.backends.cudnn.benchmark))

for label, ok in checks:
    icon = '✓' if ok else '✗'
    print(f'  [{icon}] {label}')

print('\nManual checklist (verify before every training job):')
manual = [
    'DataLoader: shuffle=True (train), shuffle=False (val)',
    'DataLoader: num_workers=4-8, pin_memory=True, persistent_workers=True',
    'DataLoader: drop_last=True (training only)',
    'Training loop: zero_grad(set_to_none=True) as first line',
    'Training loop: model.train() at epoch start, model.eval() for validation',
    'AMP: autocast(dtype=torch.bfloat16) wrapping forward + loss',
    'Gradient clipping: clip_grad_norm_(model.parameters(), 1.0)',
    'Checkpointing: tested — you can resume from an interrupted run',
]
for m in manual:
    print(f'  [ ] {m}')

---
## End-to-End: Complete Training Run (Putting It All Together)
### Slide 41: The Production-Ready Training Script

In [ ]:
# ── CELL 53 · Production training template — all segments combined ─────────────
"""
Complete production training template from Slide 41.
Every element corresponds to a lecture segment:
  - MLPClassifier        (Segment 2)
  - Six-step training    (Segment 3)
  - AdamW + cosine LR   (Segment 3)
  - CrossEntropyLoss     (Segment 3)
  - evaluate() loop      (Segment 3)
  - Best checkpoint      (Segment 3)
  - DataLoader settings  (Segment 4)
  - AMP autocast         (Segment 5)
  - gradient clipping    (Segment 3)
  - torch.compile        (Segment 5)
"""
import torch
import torch.nn as nn
import time, os, tempfile

torch.manual_seed(42)
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
TMPDIR  = tempfile.mkdtemp()

# ═══ 1. SETUP ════════════════════════════════════════════════════════════════
EPOCHS      = 10
BATCH_TRAIN = 128
BATCH_VAL   = 256
LR          = 3e-4
WD          = 0.1

# Ensure MLPClassifier is available (from Cell 18)
try:
    _ = MLPClassifier
except NameError:
    class MLPClassifier(nn.Module):
        def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.1):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
                nn.GELU(), nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
                nn.GELU(), nn.Dropout(dropout),
                nn.Linear(hidden_dim, output_dim),
            )
        def forward(self, x): return self.net(x)

model = MLPClassifier(input_dim=256, hidden_dim=512, output_dim=20).to(DEVICE)

is_compiled = False
if hasattr(torch, 'compile'):
    try:
        import torch._dynamo; torch._dynamo.reset()
        model = torch.compile(model)
        is_compiled = True
    except Exception as e:
        print(f'compile() skipped: {e}')

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WD,
    fused=(USE_AMP),     # fused kernel is faster on CUDA, unavailable on CPU
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

# ═══ 2. DATA ═════════════════════════════════════════════════════════════════
N = 4096
X_all = torch.randn(N, 256)
Y_all = torch.randint(0, 20, (N,))
split = int(N * 0.8)
train_ds = torch.utils.data.TensorDataset(X_all[:split], Y_all[:split])
val_ds   = torch.utils.data.TensorDataset(X_all[split:], Y_all[split:])

train_loader_53 = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_TRAIN, shuffle=True,
    num_workers=0, pin_memory=USE_AMP, drop_last=True,
)
val_loader_53 = torch.utils.data.DataLoader(
    val_ds, batch_size=BATCH_VAL, shuffle=False,
    num_workers=0, pin_memory=USE_AMP,
)

# ═══ 3. TRAINING LOOP ════════════════════════════════════════════════════════
best_val_loss = float('inf')
best_path     = os.path.join(TMPDIR, 'best_model.pth')
history53     = []

# Ensure evaluate() is available (from Cell 28)
try:
    _ = evaluate
except NameError:
    def evaluate(model, loader, criterion, device):
        model.eval(); total_loss = correct = total = 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                total_loss += criterion(logits, y).item() * x.size(0)
                correct += (logits.argmax(1) == y).sum().item()
                total   += x.size(0)
        model.train()
        return total_loss/total, correct/total

print(f'Training on {DEVICE}  AMP={USE_AMP}  compile={is_compiled}')
print(f'{"Ep":>3} {"tr_loss":>9} {"val_loss":>10} {"val_acc":>9} {"LR":>11} {"time":>7}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()

    model.train()
    tr_loss = 0.0
    for bx, by in train_loader_53:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=USE_AMP):
            logits = model(bx)
            loss   = criterion(logits, by)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        tr_loss += loss.item()
    scheduler.step()

    val_loss, val_acc = evaluate(model, val_loader_53, criterion, DEVICE)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Compiled models wrap the original module
        try:
            sd = model._orig_mod.state_dict()
        except AttributeError:
            sd = model.state_dict()
        torch.save(sd, best_path)
        saved = ' *'
    else:
        saved = ''

    elapsed = time.perf_counter() - t0
    lr_now  = optimizer.param_groups[0]['lr']
    avg_tr  = tr_loss / len(train_loader_53)
    print(f'{epoch:>3} {avg_tr:>9.4f} {val_loss:>10.4f} {val_acc:>9.4f} '
          f'{lr_now:>11.7f} {elapsed:>5.2f}s{saved}')
    history53.append({'epoch': epoch, 'tr_loss': avg_tr,
                      'val_loss': val_loss, 'val_acc': val_acc})

best_ep = min(history53, key=lambda r: r['val_loss'])['epoch']
print(f'\nBest checkpoint: epoch {best_ep}, val_loss={best_val_loss:.4f}')
print(f'Saved to: {best_path}')


In [ ]:
# ── CELL 54 · Training curves ─────────────────────────────────────────────────
try:
    import matplotlib.pyplot as plt
    epochs_list = [r['epoch'] for r in history53]
    tr_losses   = [r['tr_loss'] for r in history53]
    val_losses  = [r['val_loss'] for r in history53]
    val_accs    = [r['val_acc'] for r in history53]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs_list, tr_losses,  'b-o', label='Train loss', markersize=4)
    axes[0].plot(epochs_list, val_losses, 'r-o', label='Val loss',   markersize=4)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Training & Validation Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_list, val_accs, 'g-o', markersize=4)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Validation Accuracy')
    axes[1].grid(alpha=0.3)

    plt.suptitle('Week 04 — End-to-End Training Results', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/tmp/week04_training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Curves saved to /tmp/week04_training_curves.png')
except ImportError:
    print('matplotlib not installed. Training history:')
    for r in history53:
        print(f"  Epoch {r['epoch']:2d}: tr={r['tr_loss']:.4f}  val={r['val_loss']:.4f}  acc={r['val_acc']:.3f}")

In [ ]:
# ── CELL 55 · Final summary: segment-by-segment recap ────────────────────────
print('=' * 65)
print('  WEEK 04 — PYTORCH FUNDAMENTALS: WHAT WE COVERED')
print('=' * 65)

segments = [
    ('Segment 1: Tensors & Autograd',
     ['torch.randn, zeros, ones, arange — create tensors of any shape and dtype',
      '.to(device) moves tensors to GPU; operations on different devices fail',
      'requires_grad=True tells PyTorch to build a computational graph',
      'loss.backward() computes all gradients via backprop through the graph',
      'torch.no_grad() disables grad; .detach() breaks gradient flow']),
    ('Segment 2: nn.Module',
     ['Subclass nn.Module; implement __init__ and forward; call super().__init__()',
      'Always call model(x), not model.forward(x)',
      'nn.Linear, nn.Conv2d, nn.LayerNorm, nn.Dropout, nn.Embedding — core layers',
      'model.parameters() for optimiser; model.state_dict() to save',
      'Save state_dict, not the model object']),
    ('Segment 3: Training Loop',
     ['Six steps: train() -> zero_grad() -> forward -> loss -> backward -> step()',
      'AdamW(lr=3e-4, weight_decay=0.1) for modern deep learning',
      'CrossEntropyLoss takes RAW LOGITS (not softmax outputs!)',
      'model.eval() + torch.no_grad() for every validation loop',
      'Checkpoint includes model + optimizer + scheduler + scaler']),
    ('Segment 4: DataLoader',
     ['Subclass Dataset; __len__ returns count; __getitem__ returns one sample',
      'Training: shuffle=True; Validation: shuffle=False',
      'num_workers=4-8, pin_memory=True, persistent_workers=True, drop_last=True',
      'Training transforms add randomness; val transforms are deterministic',
      'GPU SM% oscillation = DataLoader starvation — fix with more workers']),
    ('Segment 5: Performance',
     ['AMP: autocast(bfloat16) = ~2x speedup + ~2x VRAM reduction in 3 lines',
      'torch.compile(model) = 20-50% speedup on Transformers in 1 line',
      'TF32 enabled by default on A100 = free ~3x matmul speedup',
      '6 common mistakes: train/eval, zero_grad, workers, device, forward(), sync',
      'Apply in order: AMP first, then compile, then profile']),
]

for seg_name, points in segments:
    print(f'\n  {seg_name}')
    for p in points:
        print(f'    • {p}')

print('\n' + '=' * 65)
print('  NEXT WEEK: Distributed Training — scaling to multiple GPUs')
print('=' * 65)